# Time-Series Trend & Rolling Metrics

This notebook demonstrates temporal data analysis, including time-series resampling (weekly/monthly), 7-day and 30-day rolling window averages, month-over-month percentage change calculations, cumulative sum tracking, and trend direction analysis for business decision-making.

### Tasks Covered:
1. **Resample Data by Time Period**: Aggregate daily data to weekly and monthly windows.
2. **Compute Rolling Window Average**: Calculate and plot 7-day and 30-day moving averages alongside raw daily revenue.
3. **Month-over-Month Percentage Change**: Compute `.pct_change()` and categorize growth vs decline months.
4. **Cumulative Sum**: Compute running revenue total and plot cumulative progression.
5. **Identify Trend Patterns & Business Implications**: Synthesize statistical trends into strategic recommendations.

## Task 1: Resample Data by Time Period

Aggregate daily revenue and order counts into weekly and monthly buckets.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Synthesize daily sales data for 2025
np.random.seed(42)
dates = pd.date_range(start='2025-01-01', end='2025-12-31', freq='D')
n = len(dates)
trend = np.linspace(8000, 14000, n)
seasonality = 1500 * np.sin(2 * np.pi * np.arange(n) / 7)
noise = np.random.normal(0, 1200, n)
revenue = np.maximum(1000.0, trend + seasonality + noise)
orders = np.random.randint(80, 250, size=n)

df = pd.DataFrame({
    'date': dates,
    'revenue': np.round(revenue, 2),
    'orders': orders
})

df_ts = df.set_index('date')
weekly_revenue = df_ts['revenue'].resample('W').sum()
weekly_count = df_ts['orders'].resample('W').count()
monthly_revenue = df_ts['revenue'].resample('ME').sum()

print('Weekly Revenue (First 5 weeks):')
print(weekly_revenue.head())
print('\nHighest Revenue Week:', weekly_revenue.idxmax().strftime('%Y-%m-%d'), f'(${weekly_revenue.max():,.2f})')
print('Highest Revenue Month:', monthly_revenue.idxmax().strftime('%Y-%m'), f'(${monthly_revenue.max():,.2f})')

## Task 2: Compute Rolling Window Average

Calculate 7-day and 30-day moving averages to smooth daily noise.

In [ ]:
df['revenue_ma7'] = df['revenue'].rolling(window=7).mean()
df['revenue_ma30'] = df['revenue'].rolling(window=30).mean()

plt.figure(figsize=(12, 6))
plt.plot(df['date'], df['revenue'], label='Raw Daily Revenue', alpha=0.3, color='gray')
plt.plot(df['date'], df['revenue_ma7'], label='7-day MA', color='blue', linewidth=1.5)
plt.plot(df['date'], df['revenue_ma30'], label='30-day MA', color='red', linewidth=2.0)
plt.title('Daily Revenue vs 7-day & 30-day Rolling Averages')
plt.xlabel('Date')
plt.ylabel('Revenue ($)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
os.makedirs('../output', exist_ok=True)
plt.savefig('../output/rolling_avg.png')
plt.show()

## Task 3: Calculate Month-over-Month Percentage Change

Compute `.pct_change()` to measure monthly growth rates.

In [ ]:
mom_change = monthly_revenue.pct_change() * 100
print('Month-over-Month Percentage Change (%):')
print(mom_change.round(2))

growth_months = mom_change[mom_change > 0]
decline_months = mom_change[mom_change < 0]
print('\nGrowth Months:', [m.strftime('%Y-%m') for m in growth_months.index])
print('Decline Months:', [m.strftime('%Y-%m') for m in decline_months.index])

## Task 4: Compute Cumulative Sum

Track total accumulated revenue over time using `.cumsum()`.

In [ ]:
df['cumulative_revenue'] = df['revenue'].cumsum()

plt.figure(figsize=(10, 5))
plt.plot(df['date'], df['cumulative_revenue'], color='green', linewidth=2.0)
plt.title('Cumulative Revenue Over Time')
plt.xlabel('Date')
plt.ylabel('Cumulative Revenue ($)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig('../output/cumulative.png')
plt.show()

print(f'Total Accumulated Revenue: ${df["cumulative_revenue"].iloc[-1]:,.2f}')

## Task 5: Identify Trend Pattern and Business Implications

Analyze recent trend direction and outline recommendations.

In [ ]:
recent_ma30 = df['revenue_ma30'].dropna().iloc[-30:]
trend_direction = 'up' if recent_ma30.iloc[-1] > recent_ma30.iloc[0] else 'down'
trend_magnitude = ((recent_ma30.iloc[-1] - recent_ma30.iloc[0]) / recent_ma30.iloc[0]) * 100

analysis = f"""
TREND ANALYSIS REPORT
=====================
Rolling Average Trend: {trend_direction.upper()}
Change over last 30 days: {trend_magnitude:.1f}%
Month-over-Month Growth (Latest Month): {mom_change.iloc[-1]:.1f}%

Business Implications:
- Sustainable upward momentum confirmed by 30-day moving average.
- Short-term daily volatility should be filtered out to avoid reactionary policy changes.
"""
print(analysis)
with open('../output/trend_analysis.txt', 'w', encoding='utf-8') as f:
    f.write(analysis)